# Solution 04 — Research Synthesis + Claim Verification

Reference implementation for Assessment 04.


## Architectural Decisions

Pipeline: **claim harvest → per-claim verify → decide → write brief**

1. Extract atomic claims relevant to the decision from sources + implied vendor pitch points.
2. Verify each claim against the corpus (supported/contested/…).
3. Decide recommendation from verified claim set (separate call to avoid prose-first bias).
4. Write executive brief constrained by the verified claim table.

Distinct from PO extraction (schema filling) and ops QA (factual lookup): emphasis on **epistemic status** of claims.


In [ ]:
import json
import os
import re
from typing import Any, Optional

# ---------------------------------------------------------------------------
# LLM configuration
# ---------------------------------------------------------------------------
# Set one of the following before running (or export in your shell):
#
#   export OPENAI_API_KEY="sk-..."
#   # optional: export OPENAI_MODEL="gpt-4o-mini"
#
#   export ANTHROPIC_API_KEY="sk-ant-..."
#   # optional: export ANTHROPIC_MODEL="claude-3-5-haiku-latest"
#
# If neither key is set, call_llm runs in MOCK mode so the notebook still
# executes for smoke-testing. Mock responses are placeholders — they are not
# a substitute for a real model during practice.
# ---------------------------------------------------------------------------

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "").strip()
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
ANTHROPIC_MODEL = os.environ.get("ANTHROPIC_MODEL", "claude-3-5-haiku-latest")

USE_MOCK = not (OPENAI_API_KEY or ANTHROPIC_API_KEY)
if USE_MOCK:
    print("⚠️  No API key found. call_llm is in MOCK mode.")
else:
    provider = "OpenAI" if OPENAI_API_KEY else "Anthropic"
    print(f"✓ Using {provider} API")


def _mock_response(prompt: str, system_prompt: str = "") -> str:
    """Lightweight deterministic mock for offline smoke tests."""
    combined = f"{system_prompt}\n{prompt}".lower()
    # Return minimal JSON-ish or text so downstream parsing can be exercised.
    if "json" in combined:
        return json.dumps({"mock": True, "note": "Replace with real API key for useful output", "items": []})
    return "[MOCK RESPONSE] Configure OPENAI_API_KEY or ANTHROPIC_API_KEY for real model output."


def call_llm(
    prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    temperature: float = 0.2,
    max_tokens: int = 2000,
) -> str:
    """Call a chat LLM. Credentials come from environment variables only."""
    if USE_MOCK:
        return _mock_response(prompt, system_prompt)

    try:
        if OPENAI_API_KEY:
            from openai import OpenAI

            client = OpenAI(api_key=OPENAI_API_KEY)
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                temperature=temperature,
                max_tokens=max_tokens,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ],
            )
            return (resp.choices[0].message.content or "").strip()

        from anthropic import Anthropic

        client = Anthropic(api_key=ANTHROPIC_API_KEY)
        resp = client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=max_tokens,
            temperature=temperature,
            system=system_prompt,
            messages=[{"role": "user", "content": prompt}],
        )
        return "".join(
            block.text for block in resp.content if getattr(block, "type", None) == "text"
        ).strip()
    except Exception as e:
        return f"[API ERROR] {type(e).__name__}: {e}"


def extract_json(text: str) -> Any:
    """Best-effort parse of a JSON object/array from model output."""
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    fence = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if fence:
        try:
            return json.loads(fence.group(1).strip())
        except json.JSONDecodeError:
            pass
    for opener, closer in (("{", "}"), ("[", "]")):
        start, end = text.find(opener), text.rfind(closer)
        if start != -1 and end > start:
            try:
                return json.loads(text[start : end + 1])
            except json.JSONDecodeError:
                continue
    raise ValueError(f"Could not parse JSON from model output:\n{text[:500]}")


# Smoke test
print("call_llm ready.")
print(call_llm("Reply with exactly: OK", system_prompt="Be terse.")[:80])


In [ ]:
SOURCES = [
  {
    "doc_id": "D1",
    "title": "Q2 Market Note \u2014 Warehouse Robotics",
    "text": "Global warehouse robotics deployments grew an estimated 18% YoY in Q2. Adoption is strongest in North America e-commerce fulfillment. Average payback period cited by mid-market buyers: 22 months. Labor shortage remains the primary buying trigger."
  },
  {
    "doc_id": "D2",
    "title": "Vendor Brief \u2014 MoveBot 3000",
    "text": "MoveBot claims 35% pick-rate improvement versus manual baselines in three pilot sites. Pilots were 8\u201312 weeks. Two of three sites were apparel; one grocery. List price starts at $48k per robot excluding integration. MoveBot has 120 production deployments globally per company blog."
  },
  {
    "doc_id": "D3",
    "title": "Independent Pilot Review \u2014 RetailCo",
    "text": "RetailCo apparel DC reported 19% pick-rate improvement after 10 weeks with MoveBot, below the vendor's 35% claim. Integration costs added ~40% to year-one spend. RetailCo notes downtime during software updates averaged 3 hours monthly."
  },
  {
    "doc_id": "D4",
    "title": "Safety Advisory \u2014 AMR Collisions",
    "text": "A regional safety board logged a 12% increase in reported AMR near-miss events YoY across all vendors. Advisories recommend denser lidar coverage in mixed pedestrian aisles. No MoveBot-specific recall is mentioned."
  },
  {
    "doc_id": "D5",
    "title": "Competitor Note \u2014 PackRunner",
    "text": "PackRunner averages $41k list and advertises 25% pick-rate gains. Independent data is scarce. PackRunner reports 200+ deployments. PackRunner lacks grocery reference customers in public materials."
  }
]

RESEARCH_QUERY = (
    "Should we run a 12-week MoveBot pilot in our apparel DC, and what claims should "
    "we treat skeptically in the vendor pitch?"
)


## Why Each Call

| Call | Why |
|------|-----|
| Harvest claims | Creates audit units before narrative |
| Verify claims | Assigns epistemic status with doc ids |
| Recommend | Decision from structured evidence, not vibes |
| Brief | Prose constrained by claim table |


In [ ]:
def synthesize_research_brief(query: str, sources: list) -> dict:
    if not query or not sources:
        return {
            "executive_brief": "Insufficient input.",
            "recommendation": "needs_more_data",
            "recommendation_rationale": "Missing query or sources.",
            "claims": [],
            "risks": [],
            "open_questions": ["Provide sources"],
            "source_coverage": {"used": [], "unused": []},
        }

    corpus = "\n\n".join(
        f"[{s['doc_id']}] {s['title']}: {s['text']}" for s in sources
    )
    all_ids = [s["doc_id"] for s in sources]

    # Call 1: harvest atomic claims
    harvest_raw = call_llm(
        prompt=(
            "List atomic factual claims relevant to the decision query, including vendor "
            "pitch claims that should be checked. Return JSON {\"claims\":[\"...\", ...]}\n\n"
            f"QUERY:\n{query}\n\nSOURCES:\n{corpus}"
        ),
        system_prompt="You extract auditable claims for research synthesis.",
        temperature=0.2,
    )
    try:
        claim_list = extract_json(harvest_raw).get("claims", [])
    except ValueError:
        claim_list = []

    # Call 2: verify each claim (batch)
    verify_raw = call_llm(
        prompt=(
            "For each claim, set status to supported|contested|unsupported|unclear using only "
            "the sources. Return JSON {\"claims\":[{claim, status, evidence_doc_ids, notes}]}\n\n"
            f"CLAIMS:\n{json.dumps(claim_list)}\n\nSOURCES:\n{corpus}"
        ),
        system_prompt="You are an evidence auditor. Never invent numbers.",
        temperature=0.0,
    )
    try:
        verified = extract_json(verify_raw).get("claims", [])
    except ValueError:
        verified = []

    used = sorted({d for c in verified for d in c.get("evidence_doc_ids", []) if d in all_ids})

    # Call 3: recommendation
    rec_raw = call_llm(
        prompt=(
            "Based on verified claims, choose recommendation: pilot | do_not_pilot | "
            "needs_more_data. Return JSON {recommendation, recommendation_rationale, risks, "
            "open_questions}.\n\n"
            f"QUERY:\n{query}\n\nVERIFIED:\n{json.dumps(verified, indent=2)[:9000]}"
        ),
        system_prompt="You are a cautious strategy researcher.",
        temperature=0.1,
    )
    try:
        rec = extract_json(rec_raw)
    except ValueError:
        rec = {
            "recommendation": "needs_more_data",
            "recommendation_rationale": "Verification parse issue",
            "risks": [],
            "open_questions": [],
        }

    # Call 4: executive brief constrained by claims
    brief_raw = call_llm(
        prompt=(
            "Write executive_brief (150-250 words) that only relies on the verified claims "
            "and recommendation. Do not add new statistics. Return JSON "
            "{\"executive_brief\":\"...\"}\n\n"
            f"QUERY:\n{query}\n\nREC:\n{json.dumps(rec)}\n\nCLAIMS:\n{json.dumps(verified)[:9000]}"
        ),
        system_prompt="You write tight decision briefs grounded in an evidence table.",
        temperature=0.2,
    )
    try:
        brief = extract_json(brief_raw).get("executive_brief", "")
    except ValueError:
        brief = brief_raw[:1200]

    unused = [i for i in all_ids if i not in used]
    return {
        "executive_brief": brief,
        "recommendation": rec.get("recommendation", "needs_more_data"),
        "recommendation_rationale": rec.get("recommendation_rationale", ""),
        "claims": verified,
        "risks": rec.get("risks", []),
        "open_questions": rec.get("open_questions", []),
        "source_coverage": {"used": used, "unused": unused},
    }


## Example Execution


In [ ]:
result = synthesize_research_brief(RESEARCH_QUERY, SOURCES)
print(json.dumps(result, indent=2)[:4000])


## Alternatives

- Summary-first then footnote: faster but smuggles unsupported claims into prose.
- Debate agents (bull/bear): good for interviews if timed; can be theatrical.
- Claim verify one-call-per-claim: higher precision, higher latency/cost.

## Tradeoffs

Four calls prioritize auditability over speed. Batch verify keeps runtime interview-friendly.

## Common Mistakes

- Treating vendor 35% as fact.
- Omitting contested status.
- Recommendation disconnected from claims.
- Inventing ROI payback not in sources.

## Beyond 45 Minutes

- Numeric claim canonicalization; citation offsets; human rating loop; bib export.
